### Notice: Please make sure json, pandas, time, and folium are installed.

In [1]:
import json

# 1. Open and read the existing JSON file
with open('transnewguinea.json', 'r') as file:
    data = json.load(file)

# 2. Add the new website information
# We can add a simple string or a nested dictionary with multiple details
data['website'] = {
    "name": "1000 Ways To Say Dog",
    "url": "https://transnewguinea.org/word/dog"
}

# 3. Write the updated dictionary back to the JSON file
# Using indent=4 makes the file easy for humans to read
with open('company_data.json', 'w') as file:
    json.dump(data, file, indent=4)

print("Website information successfully added to the file!")

Website information successfully added to the file!


In [2]:
data['website']

{'name': '1000 Ways To Say Dog', 'url': 'https://transnewguinea.org/word/dog'}

In [3]:
import pandas as pd
import requests
import time

base_url = "https://transnewguinea.org/word/dog?page="
# These are meant to work around the securities of the website.
# The point of the user agent is to act as an information buffer between the user and provider.
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Cookie": "cf_clearance=dSguBh9203G91R4hXjluMS.tlyjVs6Az2HadV1faXus-1777522182-1.2.1.1-3VTQHe37.4iZC_woanDnvfHajQ7jKol3awSo0bJa.8AfHiv.GUhQYqKENQPsWE24zS5DgVeKSNlPLbM6VWzeOdmyf405godslzAtgU0PZMNCDPJtl1qNFN_.2hsF3tAY8J_JPLYcXOUxW0PUGPblxlZ1q7g9oxjOVOXQzfn3CAfm4HZl2ZbVlB23fUp9irBWLRpUfshFzsYbTHgFny89dlR2xCMv73h9CjnCdylPumQHJkfVg_CcTxGiTGauS3sPKZJVjsar3klMVMATlcIhmUX7XDu5wHyVVHOchEl4idXP3iAdFwJYAAMgvGDnXZ2_7Uoy2Ej2Bno24zajZ5C0rA"
}

all_data = []
page = 1
max_retries = 3  # How many times to retry a page if it fails

print("Starting to extract data...")

while True:
    print(f"Fetching page {page}...")
    url = f"{base_url}{page}"
    
    page_successful = False
    
    # The Retry Loop
    for attempt in range(max_retries):
        try:
            # We added a 15-second timeout so the script doesn't hang forever
            response = requests.get(url, headers=headers, timeout=15)
            
            # If we hit the Cloudflare Timeout, wait and try again
            if response.status_code == 524:
                print(f"  Attempt {attempt + 1}: Error 524 (Server overloaded). Waiting 5s to retry...")
                time.sleep(5)
                continue
                
            # If we hit a 404 Not Found, we likely reached the end of the pages
            if response.status_code != 200:
                print(f"  Stopped. Server returned status code: {response.status_code}")
                break # Break out of the retry loop
                
            # If we get here, the page loaded perfectly!
            page_successful = True
            break 
            
        except requests.exceptions.Timeout:
            # If the request library itself times out waiting
            print(f"  Attempt {attempt + 1}: Connection timed out. Waiting 5s to retry...")
            time.sleep(5)
            
        except requests.exceptions.RequestException as e:
            print(f"  A connection error occurred: {e}")
            break
            
    # ---------------------------
    
    # If we failed all 3 retries or hit a dead end, stop the whole script
    if not page_successful:
        print("Could not fetch the page after multiple attempts. Stopping.")
        break

    # If successful, process the data just like before
    try:
        tables = pd.read_html(response.text)
        
        if not tables or len(tables[0]) == 0:
            print("No more data found. Ending loop.")
            break
            
        all_data.append(tables[0])
        page += 1
        
        # Increased the default wait time to 3 seconds to be kinder to the server
        time.sleep(3) 
        
    except ValueError:
        print("No tables found on this page. Ending loop.")
        break

# Combine and save the data
if len(all_data) > 0:
    print("\nCombining data and saving to JSON...")
    final_dataset = pd.concat(all_data, ignore_index=True)
    final_dataset.to_json("all_ways_to_say_dog.json", orient="records", indent=4)
    print(f"Success! Extracted {len(final_dataset)} words across {page-1} pages.")
else:
    print("No data could be extracted.")

Starting to extract data...
Fetching page 1...


/tmp/ipykernel_205920/3302843607.py:63: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


Fetching page 2...


/tmp/ipykernel_205920/3302843607.py:63: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


Fetching page 3...


...

Fetching page 21...


/tmp/ipykernel_205920/3302843607.py:63: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


Fetching page 22...


/tmp/ipykernel_205920/3302843607.py:63: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


Fetching page 23...
  Stopped. Server returned status code: 404
Could not fetch the page after multiple attempts. Stopping.

Combining data and saving to JSON...
Success! Extracted 1092 words across 22 pages.


In [13]:
import folium

with open('all_ways_to_say_dog.json', 'r', encoding='utf-8') as f:
    scraped_data = json.load(f)

df_scraped = pd.DataFrame(scraped_data)

# Standardize the language names for a better merge (lowercase and remove extra spaces)
# Note: Ensure 'language' matches the key in your JSON dict
df_scraped['language_clean'] = df_scraped['Language'].str.lower().str.strip()

# Load both CSV files downloaded from Glottolog
df_geo = pd.read_csv('languages_and_dialects_geo.csv')
df_languoid = pd.read_csv('languoid.csv')

# Merge the two Glottolog datasets 
# 'glottocode' in the geo file matches 'id' in the languoid file
df_glottolog = pd.merge(
    df_geo, 
    df_languoid, 
    left_on='glottocode', 
    right_on='id', 
    suffixes=('_geo', '_lang')
)

# Standardize the Glottolog language names for matching
df_glottolog['name_clean'] = df_glottolog['name_geo'].str.lower().str.strip()

# Join your scraped data with the Glottolog dataset based on the cleaned language name
df_map_data = pd.merge(
    df_scraped, 
    df_glottolog, 
    left_on='language_clean', 
    right_on='name_clean', 
    how='inner'
)

# Drop any languages that successfully matched but are missing geographic coordinates
df_map_data = df_map_data.dropna(subset=['latitude_geo', 'longitude_geo'])

# Initialize the map centered roughly over the island of New Guinea
m = folium.Map(location=[-5.0, 140.0], zoom_start=5, tiles='CartoDB positron')

# Iterate over the merged data and add markers
for idx, row in df_map_data.iterrows():
    # Create HTML for a clean, readable popup
    popup_html = f"""
    <div style="font-family: Arial, sans-serif; min-width: 150px;">
        <h4 style="margin-bottom: 5px; margin-top: 0; color: #2c3e50;">{row['Language'].title()}</h4>
        <b>Word for Dog:</b> {row['Entry']}<br>
        <hr style="margin: 5px 0;">
        <small>
        <b>Glottocode:</b> {row['glottocode']}<br>
        <b>Family:</b> {row['family_id']}
        </small>
    </div>
    """
    
    # We use CircleMarker as they look cleaner when mapping many points
    folium.CircleMarker(
        location=[row['latitude_geo'], row['longitude_geo']],
        radius=6,
        popup=folium.Popup(popup_html, max_width=300),
        color="#3186cc",
        fill=True,
        fill_color="#3186cc",
        fill_opacity=0.7
    ).add_to(m)

# Save the map to an interactive HTML file
m.save('transnewguinea_dog_map.html')
print(f"Map successfully created! Plotted {len(df_map_data)} languages.")

# Putting 'm' by itself at the end of the cell displays the map inline in Jupyter
m

Map successfully created! Plotted 567 languages.
